# PromptGFM-Bio — Kaggle Training Notebook
**Gene-Phenotype Prediction for Rare Diseases**

### ✅ Resumable Across Sessions & Accounts
This notebook saves **three Kaggle Datasets** after training so any future session — or a different Kaggle account — can skip all expensive steps:

| Dataset name (you choose) | What it stores | Skips |
|---|---|---|
| `promptgfm-model-cache` | HuggingFace BioBERT weights | ~5 min download |
| `promptgfm-data` | Raw + processed graph | ~25 min download + preprocess |
| `promptgfm-checkpoints` | Per-epoch checkpoints | training from epoch 0 |

**Setup once → add as Dataset inputs on every future session.**

> ⚙️ Before running: Settings → Accelerator → **GPU T4 x2** · Internet → **On**

## 1. Environment Check

In [1]:
import sys, subprocess, os
import torch

print(f'Python  : {sys.version}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {vram:.1f} GB  (expect ~15-16 GB on T4)')
else:
    print('⚠️  No GPU — enable in Notebook Settings → Accelerator → GPU T4')


Python  : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch : 2.9.0+cu126
CUDA    : 12.6
GPU     : Tesla T4
VRAM    : 15.6 GB  (expect ~15-16 GB on T4)


## 2. ⚙️ Session Configuration
Edit the variables below **before running any other cell**.

**`RESUME_*` flags**: set to `True` if you have added the corresponding Kaggle Dataset as input.  
**Dataset input paths**: change if you named your datasets differently.

In [2]:
# ─── RESUME FLAGS ────────────────────────────────────────────────────────
# Set True when you have added the matching dataset as notebook input
RESUME_HF_CACHE     = False  # True → skip BioBERT download (saves ~5 min)
RESUME_DATA         = True  # True → skip raw download + preprocessing (~25 min)
RESUME_CHECKPOINTS  = True  # True → resume training from last saved epoch

# ─── INPUT DATASET PATHS ─────────────────────────────────────────────────
# Kaggle mounts "Your Datasets" at /kaggle/input/datasets/USERNAME/SLUG/
# and public/competition datasets at /kaggle/input/SLUG/
# The helper below auto-detects which mount point exists.
import os as _os
_KAGGLE_USER = 'yashvermapesurr'   # ← change if using a different Kaggle account

def _find_input(slug):
    for candidate in [
        f'/kaggle/input/datasets/{_KAGGLE_USER}/{slug}',
        f'/kaggle/input/{slug}',
    ]:
        if _os.path.exists(candidate):
            return candidate
    return f'/kaggle/input/datasets/{_KAGGLE_USER}/{slug}'  # clear path even if missing

INPUT_HF_CACHE    = _find_input('promptgfm-model-cache')
INPUT_DATA        = _find_input('promptgfm-data')
INPUT_CHECKPOINTS = _find_input('promptgfm-checkpoints')

# ─── OUTPUT DATASET NAMES (used in Steps 12 + 12.5) ──────────────────────
OUTPUT_HF_CACHE    = 'promptgfm-model-cache'
OUTPUT_DATA        = 'promptgfm-data'
OUTPUT_CHECKPOINTS = 'promptgfm-checkpoints'

# ─── HF MODEL CACHE LOCATION ─────────────────────────────────────────────
HF_CACHE_DIR = '/kaggle/working/hf_cache'
_os.environ['HF_HOME']               = HF_CACHE_DIR
_os.environ['TRANSFORMERS_CACHE']    = HF_CACHE_DIR
_os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE_DIR

print('Configuration:')
print(f'  RESUME_HF_CACHE    = {RESUME_HF_CACHE}')
print(f'  RESUME_DATA        = {RESUME_DATA}')
print(f'  RESUME_CHECKPOINTS = {RESUME_CHECKPOINTS}')
print(f'  HF cache dir       = {HF_CACHE_DIR}')
print()
print('Resolved input paths:')
print(f'  INPUT_HF_CACHE    = {INPUT_HF_CACHE}  (exists={_os.path.exists(INPUT_HF_CACHE)})')
print(f'  INPUT_DATA        = {INPUT_DATA}  (exists={_os.path.exists(INPUT_DATA)})')
print(f'  INPUT_CHECKPOINTS = {INPUT_CHECKPOINTS}  (exists={_os.path.exists(INPUT_CHECKPOINTS)})')


Configuration:
  RESUME_HF_CACHE    = False
  RESUME_DATA        = True
  RESUME_CHECKPOINTS = True
  HF cache dir       = /kaggle/working/hf_cache

Resolved input paths:
  INPUT_HF_CACHE    = /kaggle/input/datasets/yashvermapesurr/promptgfm-model-cache  (exists=False)
  INPUT_DATA        = /kaggle/input/datasets/yashvermapesurr/promptgfm-data  (exists=True)
  INPUT_CHECKPOINTS = /kaggle/input/datasets/yashvermapesurr/promptgfm-checkpoints  (exists=True)


## 3. Install PyTorch Geometric

In [3]:
import torch, subprocess, sys

TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = torch.version.cuda.replace('.', '')
WHEEL_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_VER}.html'
print(f'PyG wheel source: {WHEEL_URL}')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '-f', WHEEL_URL,
     'torch-scatter', 'torch-sparse', 'torch-cluster',
     'torch-spline-conv', 'torch-geometric'],
    check=True
)
print('✅ PyTorch Geometric installed')


PyG wheel source: https://data.pyg.org/whl/torch-2.9.0+cu126.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 83.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 71.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 75.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.2 MB/s eta 0:00:00
✅ PyTorch Geometric installed


## 4. Install Extra Dependencies

In [4]:
# Upgrade build tools first — prevents metadata-generation-failed on Python 3.12
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--upgrade', 'setuptools', 'wheel', 'pip'], check=True)

extra = [
    'transformers>=4.40.0',
    'sentence-transformers>=2.7.0',
    'biopython>=1.84',
    'networkx>=3.2',
    'wandb>=0.17.0',
    'python-dotenv>=1.0.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + extra, check=True)
print('✅ Extra packages installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.9 MB/s eta 0:00:00
✅ Extra packages installed


## 5. Clone Project Code from GitHub

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Prevent git from hanging waiting for interactive credential input
os.environ['GIT_TERMINAL_PROMPT'] = '0'

GITHUB_URL  = 'https://github.com/pes1ug23am910/PromptGFM-Bio.git'
PROJECT_DIR = '/kaggle/working/PromptGFM-Bio'

# ── Try Kaggle Secret for GitHub token (needed if repo is private) ─────────
_github_token = ''
try:
    from kaggle_secrets import UserSecretsClient
    _github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass

if _github_token:
    # Embed token in URL so git doesn't need to prompt
    _auth_url = GITHUB_URL.replace('https://', f'https://{_github_token}@')
else:
    _auth_url = GITHUB_URL

if not os.path.exists(PROJECT_DIR):
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', _auth_url, PROJECT_DIR]
    )
    if result.returncode != 0:
        raise RuntimeError(
            'Git clone failed.\n'
            '  • If the repo is private: add a GitHub Personal Access Token as a\n'
            '    Kaggle Secret named "GITHUB_TOKEN" (Settings → Add-ons → Secrets)\n'
            '  • If the repo is public: check the URL is correct:\n'
            f'    {GITHUB_URL}'
        )
    print(f'✅ Cloned to {PROJECT_DIR}')
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull'])
    print(f'✅ Pulled latest changes')

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')


Cloning into '/kaggle/working/PromptGFM-Bio'...
fatal: could not read Username for 'https://github.com': No such device or address


CalledProcessError: Command '['git', 'clone', '--depth', '1', 'https://github.com/pes1ug23am910/PromptGFM-Bio.git', '/kaggle/working/PromptGFM-Bio']' returned non-zero exit status 128.

## 6. Create Directory Structure

In [ ]:
from pathlib import Path

dirs = [
    'data/raw', 'data/processed', 'data/splits',
    'checkpoints/promptgfm_film',
    'logs',
]
for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)
print('✅ Directories created')


## 7. Restore Assets from Previous Session
Restores HuggingFace model cache, preprocessed data, and training checkpoints from saved Kaggle Datasets — skipping all expensive steps below.

**First-time run**: all three blocks will print "not found — will download/preprocess fresh".

**If you see "not found" warnings when `RESUME_*=True`:**
- Make sure you added your datasets via **Add Data → Your Datasets** in the notebook sidebar **before running**.
- Check that the dataset names in cell 2 match what you created (e.g. `promptgfm-checkpoints`).
- Diagnostic lines below now print the exact path being checked and whether it exists.


In [ ]:
import shutil, tarfile
from pathlib import Path

def restore_tar(src_path, dest_dir, label):
    """Extract a .tar.gz archive if it exists."""
    src = Path(src_path)
    print(f'  checking : {src}  (exists={src.exists()})')
    if src.exists():
        dest = Path(dest_dir)
        dest.mkdir(parents=True, exist_ok=True)
        with tarfile.open(src, 'r:gz') as tar:
            tar.extractall(dest)
        print(f'  ✅ {label} restored from {src}')
        return True
    return False

def restore_dir(src_path, dest_dir, label):
    """Copy directory tree if source exists."""
    src = Path(src_path)
    print(f'  checking : {src}  (exists={src.exists()})')
    if src.exists() and any(src.iterdir()):
        dest = Path(dest_dir)
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(src, dest)
        print(f'  ✅ {label} restored from {src}')
        return True
    return False

# ── A. HuggingFace model cache ────────────────────────────────────────────
if RESUME_HF_CACHE:
    print(f'[HF cache] Input path: {INPUT_HF_CACHE}  (exists={Path(INPUT_HF_CACHE).exists()})')
    ok = restore_tar(f'{INPUT_HF_CACHE}/hf_cache.tar.gz', HF_CACHE_DIR, 'HF model cache')
    if not ok:
        ok = restore_dir(f'{INPUT_HF_CACHE}/hf_cache', HF_CACHE_DIR, 'HF model cache')
    if not ok:
        print(f'⚠️  HF cache not found at {INPUT_HF_CACHE} — BioBERT will be re-downloaded')
else:
    print('HF cache: skipped (RESUME_HF_CACHE=False)')

# ── B. Preprocessed graph + raw data ─────────────────────────────────────
if RESUME_DATA:
    print(f'\n[Data] Input path: {INPUT_DATA}  (exists={Path(INPUT_DATA).exists()})')
    if Path(INPUT_DATA).exists():
        print(f'  contents: {[p.name for p in Path(INPUT_DATA).iterdir()]}')
    ok = restore_tar(f'{INPUT_DATA}/data.tar.gz', 'data', 'Graph data')
    if not ok:
        ok = restore_dir(f'{INPUT_DATA}/processed', 'data/processed', 'Processed graph')
        restore_dir(f'{INPUT_DATA}/splits', 'data/splits', 'Data splits')
    graph = Path('data/processed/biomedical_graph.pt')
    if graph.exists():
        print(f'✅ Graph ready ({graph.stat().st_size/1e6:.0f} MB)')
    else:
        print('⚠️  Graph not found in restored data — will preprocess fresh')
        RESUME_DATA = False   # force re-preprocessing below
else:
    print('Data: skipped (RESUME_DATA=False)')

# ── C. Training checkpoints ───────────────────────────────────────────────
if RESUME_CHECKPOINTS:
    ckpt_src = Path(INPUT_CHECKPOINTS)
    ckpt_dst = Path('checkpoints/promptgfm_film')
    ckpt_dst.mkdir(parents=True, exist_ok=True)
    print(f'\n[Checkpoints] Input path: {ckpt_src}  (exists={ckpt_src.exists()})')
    if ckpt_src.exists():
        print(f'  contents: {[p.name for p in ckpt_src.iterdir()]}')
    files_copied = 0
    if ckpt_src.exists():
        # Copy all files — use rglob to handle any subdirectory nesting
        for f in ckpt_src.rglob('*'):
            if f.is_file():
                shutil.copy2(f, ckpt_dst / f.name)
                files_copied += 1
    if files_copied:
        epochs = sorted([f.stem.replace('checkpoint_epoch_','') for f in ckpt_dst.glob('checkpoint_epoch_*.pt')])
        print(f'✅ Checkpoints restored ({files_copied} files). Epochs available: {epochs}')
    else:
        print(f'⚠️  No checkpoints found at {INPUT_CHECKPOINTS} — will train from scratch')
        RESUME_CHECKPOINTS = False
else:
    print('Checkpoints: skipped (RESUME_CHECKPOINTS=False)')


## 7.5. Pre-download BioBERT Model
Downloads the full **440 MB** PubMedBERT weights to disk **before** training.

**Why this cell exists:** The model uses safetensors lazy-loading, which streams weights from the network one parameter at a time. If Kaggle's connection to HuggingFace CDN hiccups mid-stream (~31% is a common stall point), the training cell hangs indefinitely.

Running `snapshot_download` here downloads the complete file to disk first.  
**If this cell stalls:** click ■ Stop → re-run only this cell — it resumes from where it left off.

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import warnings, os

MODEL_NAME = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext'

hub_dir = Path(HF_CACHE_DIR) / 'hub'
model_cache_name = 'models--' + MODEL_NAME.replace('/', '--')
model_snapshots = hub_dir / model_cache_name / 'snapshots'

if RESUME_HF_CACHE and model_snapshots.exists() and any(model_snapshots.iterdir()):
    print('⏭️  BioBERT already in restored HF cache — skipping download')
else:
    print(f'Downloading {MODEL_NAME}')
    print(f'  Cache dir : {HF_CACHE_DIR}')
    print(f'  Size      : ~440 MB — takes ~2-3 min on Kaggle')
    print(f'  Downloads resume automatically if interrupted\n')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UserWarning)
        snapshot_download(
            repo_id=MODEL_NAME,
            cache_dir=HF_CACHE_DIR,
            ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*', 'rust_model*', '*.ot'],
        )
    print('\n✅ BioBERT fully downloaded and cached')

# ── Force offline mode so training NEVER touches the network for model weights ──
# This prevents from_pretrained() making freshness-check HEAD requests that can
# stall indefinitely on Kaggle's flaky HuggingFace CDN connection.
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE']       = '1'
print('✅ Offline mode enabled — training will load BioBERT from disk only')


## 8. Download Biomedical Datasets
Skipped automatically if `RESUME_DATA=True` and graph was restored successfully.  
Total download: ~1.5 GB · takes ~10 min.

In [ ]:
from pathlib import Path

graph_exists = Path('data/processed/biomedical_graph.pt').exists()

if RESUME_DATA and graph_exists:
    print('⏭️  Download skipped — restored from Kaggle Dataset')
else:
    print('Downloading datasets...')
    result = subprocess.run(
        [sys.executable, 'scripts/download_data.py', '--dataset', 'all'],
        capture_output=False
    )
    print('Download exit code:', result.returncode)


## 9. Preprocess Data (Build Knowledge Graph)
Skipped automatically if `RESUME_DATA=True` and graph was restored successfully.

In [ ]:
from pathlib import Path

graph_path = Path('data/processed/biomedical_graph.pt')

if RESUME_DATA and graph_path.exists():
    print(f'⏭️  Preprocessing skipped — graph ready ({graph_path.stat().st_size/1e6:.0f} MB)')
else:
    print('Building knowledge graph...')
    result = subprocess.run(
        [sys.executable, 'scripts/preprocess_all.py'],
        capture_output=False
    )
    print('Preprocessing exit code:', result.returncode)
    if graph_path.exists():
        print(f'✅ Graph ready ({graph_path.stat().st_size/1e6:.0f} MB)')
    else:
        raise RuntimeError('Graph file not created — check logs above')


## 9.5. 💾 Save Graph Data Now
Saves the graph immediately after building it — **before training starts**.

This means if training crashes or the session times out, the expensive preprocessing work is not lost. Next session you can set `RESUME_DATA=True` and jump straight to training.

> Skipped automatically if `RESUME_DATA=True` (graph was already restored from a saved dataset).

In [ ]:
import tarfile
from pathlib import Path

if RESUME_DATA:
    print('⏭️  Graph was restored from saved dataset — no need to re-save')
else:
    _out = Path('/kaggle/working/out_data/data.tar.gz')
    _out.parent.mkdir(parents=True, exist_ok=True)
    _saved = []
    with tarfile.open(_out, 'w:gz') as _tar:
        for _d in ['data/processed', 'data/splits']:
            _p = Path(_d)
            if _p.exists() and any(_p.rglob('*')):
                _tar.add(_p, arcname=_p.name)
                _saved.append(_p.name)
    if _saved:
        print(f'✅ Saved {", ".join(_saved)} → {_out}  ({_out.stat().st_size/1e6:.0f} MB)')
        print()
        print('📌 To preserve this for future sessions:')
        print('   Output tab (right panel) → out_data/ → ⊕ → New Dataset → name it "promptgfm-data"')
        print('   Next session: add that dataset as input and set RESUME_DATA=True in cell 2')
    else:
        print('⚠️  Nothing to save — data/processed and data/splits are empty')


## 10. W&B Login (Optional)

In [ ]:
# ── Try Kaggle Secrets first, then fall back to empty (W&B disabled) ──────
WANDB_API_KEY = ''
try:
    from kaggle_secrets import UserSecretsClient
    WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    pass  # no secret configured — W&B will be disabled

# WANDB_API_KEY = ''   # uncomment to force-disable W&B

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print('✅ W&B logged in')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B disabled — metrics logged to stdout only')


## 11. Train
Uses `configs/kaggle_config.yaml` (T4-tuned: batch_size=256, hidden_dim=512).

**Speed improvements active in this version:**
- 🚀 Frozen BioBERT embeddings are pre-computed once before training (eliminates per-batch BERT inference)
- 🚀 GNN embeddings cached during validation (reduces validation time ~30 %)
- 🚀 Dual-GPU (T4 ×2): `torchrun --nproc_per_node=2` is used automatically when 2 GPUs are detected

Set `RESUME_CHECKPOINTS=True` in cell 2 to continue from the last restored checkpoint.

In [ ]:
import os
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE']       = '1'
print('✅ Offline mode confirmed — BioBERT loads from disk only')


In [ ]:
import subprocess, os
os.environ['GIT_TERMINAL_PROMPT'] = '0'
result = subprocess.run(['git', '-C', '/kaggle/working/PromptGFM-Bio', 'pull'])
if result.returncode == 0:
    print('✅ Latest code pulled from GitHub')
else:
    print('⚠️  git pull failed — continuing with existing code (this is OK if repo is private)')


In [ ]:
import torch as _torch
from pathlib import Path as _Path

# --- Auto-detect in-session checkpoints for same-session restart ---
_ckpt_dir = _Path("checkpoints/promptgfm_film")
_ckpts = sorted(
    _ckpt_dir.glob("checkpoint_epoch_*.pt"),
    key=lambda p: int(p.stem.split("_")[-1])
) if _ckpt_dir.exists() else []

# RESUME_CHECKPOINTS=True  = resuming from a Kaggle Dataset (next session)
# _ckpts non-empty         = in-session restart (same kernel, stopped+restarted)
RESUME = RESUME_CHECKPOINTS or bool(_ckpts)

# Always use train.py  (resume_training.py is interactive and hangs in notebooks)
base_args = ["scripts/train.py", "--config", "configs/kaggle_config.yaml"]
if RESUME and _ckpts:
    _last_ckpt = str(_ckpts[-1])
    base_args += ["--resume-checkpoint", _last_ckpt]
    print(f"In-session resume from: {_last_ckpt}")
elif RESUME:
    print("RESUME_CHECKPOINTS=True but no local checkpoints found -- starting fresh")
    RESUME = False

# --- Multi-GPU: use torchrun when 2+ GPUs are available ---
n_gpus = _torch.cuda.device_count()
print(f"GPUs available: {n_gpus}")

if n_gpus > 1:
    import shutil as _shutil
    _torchrun = _shutil.which("torchrun") or "torchrun"
    cmd = [_torchrun, f"--nproc_per_node={n_gpus}", "--master_port=29500"] + base_args
    print(f"Launching DDP on {n_gpus} x T4")
else:
    cmd = [sys.executable] + base_args
    print("Single-GPU launch")

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=False)
print("Training exit code:", result.returncode)


## 12. 💾 Save ALL Assets as Kaggle Output

Run this cell **before the session ends** (set a reminder before the 9-hour limit).

It saves three directories under `/kaggle/working/`:

| Directory | Contents | Create Dataset named |
|---|---|---|
| `out_model_cache/` | BioBERT weights (~440 MB) | `promptgfm-model-cache` |
| `out_data/` | Raw + processed graph (~600 MB) | `promptgfm-data` |
| `out_checkpoints/` | Per-epoch `.pt` files | `promptgfm-checkpoints` |

### After this cell completes:
1. Click **Output** tab (right panel) → you'll see these three folders
2. For **each** folder → click the ⊕ icon → **New Dataset** → use the names above
3. Make the datasets **Public** (or **Private** if you want them only for yourself)
4. Next session: **Add Data** → **Your Datasets** → add all three → set `RESUME_*=True`

### Using from a different Kaggle account:
Make the datasets **Public**, then the other account can find them by searching  
`pes1ug23am910/promptgfm-model-cache` etc. in **Add Data**.

In [ ]:
import shutil, tarfile, os
from pathlib import Path

def make_tar(src_dir, out_file, label):
    src = Path(src_dir)
    if not src.exists() or not any(src.rglob('*')):
        print(f'⚠️  {label}: source empty or missing ({src}) — skipped')
        return
    out = Path(out_file)
    out.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(out, 'w:gz') as tar:
        tar.add(src, arcname=src.name)
    size_mb = out.stat().st_size / 1e6
    print(f'✅ {label} → {out}  ({size_mb:.0f} MB)')

def copy_dir(src_dir, out_dir, label):
    src = Path(src_dir)
    out = Path(out_dir)
    if not src.exists():
        print(f'⚠️  {label}: missing ({src}) — skipped')
        return
    if out.exists():
        shutil.rmtree(out)
    shutil.copytree(src, out)
    files = list(out.rglob('*.*'))
    total_mb = sum(f.stat().st_size for f in files) / 1e6
    print(f'✅ {label} → {out}  ({len(files)} files, {total_mb:.0f} MB)')

print('=== Saving assets for next session ===')

# A. HuggingFace model cache (BioBERT) ─────────────────────────────────────
make_tar(HF_CACHE_DIR,
         '/kaggle/working/out_model_cache/hf_cache.tar.gz',
         'HF model cache')

# B. Data — processed graph + splits bundled together ──────────────────────
# Saves BOTH data/processed/ and data/splits/ so RESUME_DATA=True skips
# both the 10-min download (step 8) AND the graph build (step 9) next session.
_data_out = Path('/kaggle/working/out_data/data.tar.gz')
_data_out.parent.mkdir(parents=True, exist_ok=True)
_saved = []
with tarfile.open(_data_out, 'w:gz') as _tar:
    for _d in ['data/processed', 'data/splits']:
        _p = Path(_d)
        if _p.exists() and any(_p.rglob('*')):
            _tar.add(_p, arcname=_p.name)   # stores as 'processed/' + 'splits/' at tar root
            _saved.append(_p.name)
if _saved:
    print(f'✅ Graph data ({", ".join(_saved)}) → {_data_out}  ({_data_out.stat().st_size/1e6:.0f} MB)')
else:
    _data_out.unlink(missing_ok=True)
    print('⚠️  Graph data: nothing to save — run steps 8 and 9 first')

# C. Training checkpoints ──────────────────────────────────────────────────
copy_dir('checkpoints/promptgfm_film',
         '/kaggle/working/out_checkpoints',
         'Training checkpoints')

print()
print('=== Output summary ===')
for d in ['out_model_cache', 'out_data', 'out_checkpoints']:
    p = Path('/kaggle/working') / d
    if p.exists():
        files = list(p.rglob('*.*'))
        total_mb = sum(f.stat().st_size for f in files) / 1e6
        print(f'  /kaggle/working/{d}/  →  {len(files)} files, {total_mb:.0f} MB')

print()
print('Next steps:')
print('  1. Output tab → create 3 datasets from out_model_cache, out_data, out_checkpoints')
print('  2. Next session: add those datasets as input, set RESUME_*=True in cell 2')


## 12.5. 🚀 Auto-Push All Datasets to Kaggle

Automatically pushes the three output folders to your Kaggle Datasets as new versions.

**Requirements:**
- Run **Step 12** first (prepares the output folders)
- The three datasets must already exist on your Kaggle account (created once via Output tab)
- Kaggle CLI credentials are pre-configured automatically on Kaggle notebooks

> If a push fails, fall back to: **Output tab → folder → ⊕ → New Dataset Version**


In [ ]:
import subprocess, json
from pathlib import Path

# ── Kaggle username ────────────────────────────────────────────────────────
# Auto-detected from the pre-configured Kaggle credentials on this notebook.
# Override here if you need to push to a different account.
KAGGLE_USERNAME = 'yashvermapesurr'
try:
    _cred = json.loads((Path.home() / '.kaggle' / 'kaggle.json').read_text())
    KAGGLE_USERNAME = _cred.get('username', KAGGLE_USERNAME)
except Exception:
    pass
print(f'Kaggle username : {KAGGLE_USERNAME}')

# ── Write dataset-metadata.json into each output folder ───────────────────
# Required by `kaggle datasets version` to know which dataset to update.
_meta_map = {
    '/kaggle/working/out_checkpoints': (OUTPUT_CHECKPOINTS, 'PromptGFM Checkpoints'),
    '/kaggle/working/out_data':        (OUTPUT_DATA,        'PromptGFM Data'),
    '/kaggle/working/out_model_cache': (OUTPUT_HF_CACHE,    'PromptGFM Model Cache'),
}
for _dir, (_slug, _title) in _meta_map.items():
    _p = Path(_dir)
    if _p.exists():
        _meta = _p / 'dataset-metadata.json'
        if not _meta.exists():
            _meta.write_text(json.dumps({
                'title': _title,
                'id': f'{KAGGLE_USERNAME}/{_slug}',
                'licenses': [{'name': 'CC0-1.0'}],
            }, indent=2))

# ── Commit message: include last trained epoch ─────────────────────────────
_ckpts = sorted(
    Path('checkpoints/promptgfm_film').glob('checkpoint_epoch_*.pt'),
    key=lambda _f: int(_f.stem.split('_')[-1])
) if Path('checkpoints/promptgfm_film').exists() else []
_last_epoch = _ckpts[-1].stem.split('_')[-1] if _ckpts else '?'

# ── Push each output directory as a new dataset version ───────────────────
_push_jobs = [
    ('/kaggle/working/out_checkpoints', f'epoch {_last_epoch} checkpoints'),
    ('/kaggle/working/out_data',        'graph data'),
    ('/kaggle/working/out_model_cache', 'hf model cache'),
]

print()
for _out_dir, _label in _push_jobs:
    _p = Path(_out_dir)
    if not _p.exists() or not any(_p.rglob('*.*')):
        print(f'⚠️  {_label} : {_out_dir} is empty — run Step 12 first')
        continue
    print(f'Pushing  {_label}  ({_out_dir}) ...')
    _res = subprocess.run(
        ['kaggle', 'datasets', 'version',
         '-p', _out_dir,
         '--dir-mode', 'tar',
         '-m', _label],
        capture_output=False,
    )
    if _res.returncode == 0:
        print(f'  ✅ pushed as new version')
    else:
        print(f'  ❌ failed (exit {_res.returncode})')
        print(f'     Manual fallback: Output tab → {_p.name}/ → ⊕ → New Dataset Version')

print()
print('✅ Done. Next session: set RESUME_*=True in cell 2.')


## 13. Quick Evaluation

In [ ]:
from pathlib import Path

best = Path('checkpoints/promptgfm_film/best_model.pt')
if not best.exists():
    print('No best_model.pt yet — run more training epochs first')
else:
    result = subprocess.run(
        [sys.executable, 'scripts/evaluate.py',
         '--config', 'configs/kaggle_config.yaml',
         '--checkpoint', str(best)],
        capture_output=False
    )
    print('Evaluation exit code:', result.returncode)
